# Нагрузочное тестирование сервиса

Вся логика — в модуле `load_test.py` (запускается и как скрипт из CI:
`python load_test.py`, код возврата — статус SLO). Здесь — тот же прогон
в интерактивном виде. Сервис должен быть поднят (`uvicorn app1:app --port 8079`
или docker compose). Параметры — переменными окружения: `LOAD_TEST_URL`,
`LOAD_TEST_WORKERS`, `LOAD_TEST_MAX_TIME` (секунды), `LOAD_TEST_SAMPLES`,
`LOAD_TEST_P95_SLO_MS`, `LOAD_TEST_MIN_SUCCESS_RATE`, `LOAD_TEST_DATA_PATH`,
`LOAD_TEST_FIRST_ROWS`, `LOAD_TEST_REPORT_DIR`. Отчёты пишутся в
`artifacts/load_test_report.html` / `artifacts/load_test_report.png`;
при нарушении SLO ячейка падает с AssertionError.

In [ ]:
from IPython.display import HTML, display
import pandas as pd

import load_test

label_map = load_test.load_label_classes()
profiles = load_test.load_profiles()
print(f'URL: {load_test.URL}, workers: {load_test.WORKERS}, '
      f'max_time: {load_test.MAX_TIME}с, профилей: {len(profiles)}')
print(f'SLO: p95 < {load_test.P95_SLO_MS} мс, '
      f'success_rate >= {load_test.MIN_SUCCESS_RATE}%')

In [ ]:
# Прогон. Для повторного/долгого прогона: LOAD_TEST_MAX_TIME=505,
# LOAD_TEST_WORKERS=50 (CODE_REVIEW §5.2 — параметры через env).
print('Starting load test...')
results = load_test.run_load_test(profiles, load_test.URL,
                                max_time=load_test.MAX_TIME,
                                workers=load_test.WORKERS,
                                label_map=label_map)
print('Test completed, generating report...')
report = load_test.generate_report(results)

display(HTML(pd.DataFrame([report]).to_html()))
display(HTML('<h2>First 5 responses:</h2>'))
display(results.head().to_html())